In [ ]:
# Notebook cell: bronze_snapshot_linear_freshdesk.ipynb
# Purpose: Create bronze snapshots for Linear and Freshdesk.
# - On first run: backfill ~30 days (cutoff = now - 30 days)
# - Subsequent runs: incremental since last snapshot (based on data CSV or metadata)
# - Save raw JSON to raw/<system>/raw_{TIMESTAMP}.json
# - Append normalized index rows to data/<system>/*.csv (atomic writes)
#
# Requirements: pip install requests pandas python-dateutil


import requests
import json
import time
import tempfile
import shutil
from pathlib import Path
from datetime import datetime, timezone, timedelta
import pandas as pd
from dateutil import parser as dateparse
import sys
import traceback

# -------------------------
# CONFIG 
# -------------------------
PROJECT_ROOT = Path(r"C:\Users\michael.brostrom\Documents\GitHub\statistik-freshdesk-linear")
RAW_DIR = PROJECT_ROOT / "raw"
DATA_DIR = PROJECT_ROOT / "data"
LOGS_DIR = PROJECT_ROOT / "logs"
CREDENTIALS_DIR = PROJECT_ROOT / "credentials"

# Linear
LINEAR_API_KEY_PATH = CREDENTIALS_DIR / "Linear_API-key.txt"
LINEAR_API_URL = "https://api.linear.app/graphql"
# GraphQL page size
LINEAR_PAGE_SIZE = 100

# Freshdesk
FRESHDESK_API_KEY_PATH = CREDENTIALS_DIR / "Freshdesk_API-key.txt"
FRESHDESK_DOMAIN = "intersolia.freshdesk.com"
FRESHDESK_TICKETS_ENDPOINT = f"https://{FRESHDESK_DOMAIN}/api/v2/tickets"
FRESHDESK_PAGE_SIZE = 100

# Backfill window for first run
BACKFILL_DAYS = 30

# File names
LINEAR_RAW_SUBDIR = RAW_DIR / "linear"
FRESHDESK_RAW_SUBDIR = RAW_DIR / "freshdesk"
DATA_LINEAR_CSV = DATA_DIR / "linear" / "linear_issues.csv"
DATA_FRESHDESK_CSV = DATA_DIR / "freshdesk" / "freshdesk_tickets.csv"

# Metadata files to track last snapshot
META_DIR = PROJECT_ROOT / "meta"
META_DIR.mkdir(parents=True, exist_ok=True)
LINEAR_LAST_SNAPSHOT = META_DIR / "linear_last_snapshot.txt"
FRESHDESK_LAST_SNAPSHOT = META_DIR / "freshdesk_last_snapshot.txt"

# Ensure directories exist
for p in [RAW_DIR, DATA_DIR, LOGS_DIR, LINEAR_RAW_SUBDIR, FRESHDESK_RAW_SUBDIR, DATA_DIR / "linear", DATA_DIR / "freshdesk"]:
    p.mkdir(parents=True, exist_ok=True)

# -------------------------
# Helpers
# -------------------------

def http_post_with_retry(url, headers=None, json_payload=None, max_retries=3, backoff=1.0, timeout=60):
    for attempt in range(1, max_retries+1):
        try:
            r = requests.post(url, headers=headers, json=json_payload, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(backoff * (2 ** (attempt-1)))

def http_get_with_retry(url, auth=None, params=None, max_retries=3, backoff=1.0, timeout=60):
    for attempt in range(1, max_retries+1):
        try:
            r = requests.get(url, auth=auth, params=params, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(backoff * (2 ** (attempt-1)))

def utc_now_ts():
    return datetime.now(timezone.utc)

def ts_for_filename(dt=None):
    if dt is None:
        dt = utc_now_ts()
    return dt.strftime("%Y%m%dT%H%M%SZ")

def atomic_write_text(path: Path, text: str, encoding="utf-8"):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding=encoding)
    tmp.replace(path)

def atomic_write_bytes(path: Path, b: bytes):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(b)
    tmp.replace(path)

def read_api_key(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing API key file: {path}")
    key = path.read_text(encoding="utf-8").strip()
    if not key:
        raise ValueError(f"API key file {path} is empty")
    return key

def read_last_snapshot(path: Path):
    if path.exists():
        txt = path.read_text(encoding="utf-8").strip()
        try:
            return dateparse.parse(txt)
        except Exception:
            return None
    return None

def write_last_snapshot(path: Path, dt: datetime):
    atomic_write_text(path, dt.astimezone(timezone.utc).isoformat())

def append_csv_atomic(target: Path, df: pd.DataFrame):
    target.parent.mkdir(parents=True, exist_ok=True)
    # Write to temp file first
    tmp = target.with_suffix(".tmp")
    # If file exists, append without header; else write header
    if target.exists():
        df.to_csv(tmp, index=False, mode='w', header=False)
        # Append tmp to target atomically by opening target in append mode
        with open(target, "ab") as f_target, open(tmp, "rb") as f_tmp:
            f_target.write(f_tmp.read())
        tmp.unlink(missing_ok=True)
    else:
        df.to_csv(tmp, index=False, mode='w', header=True)
        tmp.replace(target)

# -------------------------
# Linear fetch + normalize
# -------------------------
def fetch_linear_issues(api_key: str, cutoff_dt: datetime = None):
    """
    Fetch all issues from Linear (paginated). If cutoff_dt provided, filter locally
    to issues where createdAt or updatedAt >= cutoff_dt.
    Returns list of issue dicts (raw GraphQL nodes).
    """
    headers = {
        "Authorization": api_key,  # Linear expects key alone
        "Content-Type": "application/json",
        "Accept": "application/json",
        "User-Agent": "linear-bronze-snapshot/1.0"
    }

    # Build a safe selection of fields (minimal index + nested selections)
    # These fields are robust per earlier introspection
    sel = """
        id
        number
        identifier
        title
        description
        createdAt
        updatedAt
        archivedAt
        priority
        estimate
        trashed
        state { id name type }
        assignee { id name email }
        project { id name }
        team { id name }
        labels(first:50) { nodes { id name } }
    """

    query = f"""
    query($first:Int!, $after:String) {{
      issues(first:$first, after:$after) {{
        nodes {{
          {sel}
        }}
        pageInfo {{ hasNextPage endCursor }}
      }}
    }}
    """

    issues = []
    cursor = None
    page = 0
    while True:
        page += 1
        payload = {"query": query, "variables": {"first": LINEAR_PAGE_SIZE, "after": cursor}}
        resp = http_post_with_retry(LINEAR_API_URL, headers=headers, json_payload=payload, max_retries=3, backoff=1.0, timeout=60)
        data = resp.json()
        if "errors" in data:
            err_text = json.dumps(data["errors"], ensure_ascii=False)
            if "Cannot query field" in err_text or "Unknown field" in err_text:
                raise RuntimeError(f"GraphQL selection invalid for this schema: {err_text}")
            raise RuntimeError(f"Linear GraphQL errors: {data['errors']}")

        block = data["data"]["issues"]["nodes"]
        # Optionally filter by cutoff_dt
        if cutoff_dt:
            filtered = []
            for it in block:
                # prefer createdAt, fallback to updatedAt
                created = None
                updated = None
                try:
                    created = dateparse.parse(it.get("createdAt")) if it.get("createdAt") else None
                except Exception:
                    created = None
                try:
                    updated = dateparse.parse(it.get("updatedAt")) if it.get("updatedAt") else None
                except Exception:
                    updated = None
                if (created and created >= cutoff_dt) or (updated and updated >= cutoff_dt):
                    filtered.append(it)
            issues.extend(filtered)
        else:
            issues.extend(block)
        page_info = data["data"]["issues"]["pageInfo"]
        if page_info.get("hasNextPage"):
            cursor = page_info.get("endCursor")
            time.sleep(0.2)
        else:
            break
    return issues

# Normalize_linear_issue 
def normalize_linear_issue(node, snapshot_ts_iso, raw_path):
    labels_nodes = (node.get("labels") or {}).get("nodes", [])
    labels = ";".join([lbl.get("name","") for lbl in labels_nodes])
    state = node.get("state") or {}
    assignee = node.get("assignee") or {}
    project = node.get("project") or {}
    team = node.get("team") or {}

    return {
        "snapshot_ts": snapshot_ts_iso,
        "id": node.get("id"),
        "number": node.get("number"),
        "identifier": node.get("identifier"),
        "title": node.get("title"),
        "description": node.get("description"),
        "createdAt": node.get("createdAt"),
        "updatedAt": node.get("updatedAt"),
        "archivedAt": node.get("archivedAt"),
        "priority": node.get("priority"),
        "estimate": node.get("estimate"),
        "trashed": node.get("trashed"),
        "state_id": state.get("id"),
        "state_name": state.get("name"),
        "state_type": state.get("type"),
        "assignee_id": assignee.get("id"),
        "assignee_name": assignee.get("name"),
        "assignee_email": assignee.get("email"),
        "project_id": project.get("id"),
        "project_name": project.get("name"),
        "team_id": team.get("id"),
        "team_name": team.get("name"),
        "labels": labels,
        "raw_json_path": str(raw_path)
    }

# -------------------------
# Freshdesk fetch + normalize
# -------------------------
def fetch_freshdesk_tickets(api_key: str, cutoff_dt: datetime = None):
    """
    Fetch tickets from Freshdesk.
    If cutoff_dt provided, use updated_since param to fetch only tickets updated since cutoff.
    Returns list of ticket dicts.
    """

    auth = (api_key, "X")
    tickets = []
    page = 1
    params = {"per_page": FRESHDESK_PAGE_SIZE, "page": page}
    if cutoff_dt:
        # Freshdesk expects ISO8601 without microseconds, e.g. 2026-05-29T11:31:54Z
        params["updated_since"] = cutoff_dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")
    while True:
        params["page"] = page
        resp = http_get_with_retry(FRESHDESK_TICKETS_ENDPOINT, auth=auth, params=params, max_retries=3, backoff=1.0, timeout=60)
        block = resp.json()
        if not isinstance(block, list):
            # Freshdesk may return dict on error
            raise RuntimeError(f"Unexpected Freshdesk response: {block}")
        tickets.extend(block)
        # Pagination: if less than page size, we're done
        if len(block) < FRESHDESK_PAGE_SIZE:
            break
        page += 1
        time.sleep(0.2)
    # Optionally filter locally by created/updated timestamps if cutoff_dt provided
    if cutoff_dt:
        filtered = []
        for t in tickets:
            updated = None
            created = None
            try:
                updated = dateparse.parse(t.get("updated_at")) if t.get("updated_at") else None
            except Exception:
                updated = None
            try:
                created = dateparse.parse(t.get("created_at")) if t.get("created_at") else None
            except Exception:
                created = None
            if (updated and updated >= cutoff_dt) or (created and created >= cutoff_dt):
                filtered.append(t)
        return filtered
    return tickets

def normalize_freshdesk_ticket(t, snapshot_ts_iso, raw_path):
    # Map common fields; Freshdesk fields vary by account
    return {
        "snapshot_ts": snapshot_ts_iso,
        "ticket_id": t.get("id"),
        "subject": t.get("subject"),
        "description": t.get("description"),
        "created_at": t.get("created_at"),
        "updated_at": t.get("updated_at"),
        "status_id": t.get("status"),
        "status_name": None,  # status name mapping can be added later if needed
        "priority": t.get("priority"),
        "requester_id": t.get("requester_id"),
        "requester_name": None,
        "assignee_id": t.get("responder_id"),
        "assignee_name": None,
        "tags": ";".join(t.get("tags", [])) if t.get("tags") else "",
        "raw_json_path": str(raw_path)
    }

# -------------------------
# Main snapshot logic
# -------------------------
def run_snapshot():
    start_time = utc_now_ts()
    ts = ts_for_filename(start_time)
    snapshot_ts_iso = start_time.isoformat()

    log = {"start": snapshot_ts_iso, "linear": {}, "freshdesk": {}, "errors": []}

    # --- Linear ---
    try:
        linear_key = read_api_key(LINEAR_API_KEY_PATH)
        # Determine cutoff: last snapshot or backfill
        last_linear = read_last_snapshot(LINEAR_LAST_SNAPSHOT)
        if last_linear is None:
            cutoff_linear = utc_now_ts() - timedelta(days=BACKFILL_DAYS)
            first_run = True
        else:
            cutoff_linear = last_linear
            first_run = False

        print(f"[Linear] cutoff = {cutoff_linear.isoformat()} (first_run={first_run})")

        linear_issues = fetch_linear_issues(linear_key, cutoff_dt=cutoff_linear)
        print(f"[Linear] fetched {len(linear_issues)} issues (post-filter)")

        # Save raw JSON
        raw_linear_path = LINEAR_RAW_SUBDIR / f"linear_raw_{ts}.json"
        atomic_write_text(raw_linear_path, json.dumps(linear_issues, indent=2, ensure_ascii=False))
        print(f"[Linear] saved raw JSON to {raw_linear_path}")

        # Normalize and append to data CSV
        rows = [normalize_linear_issue(it, snapshot_ts_iso, raw_linear_path) for it in linear_issues]
        df_linear = pd.DataFrame(rows)
        if not df_linear.empty:
            append_csv_atomic(DATA_LINEAR_CSV, df_linear)
            print(f"[Linear] appended {len(df_linear)} rows to {DATA_LINEAR_CSV}")
        else:
            print("[Linear] no rows to append")

        # Update last snapshot metadata
        write_last_snapshot(LINEAR_LAST_SNAPSHOT, start_time)
        log["linear"]["count"] = len(linear_issues)
    except Exception as e:
        tb = traceback.format_exc()
        print("[Linear] ERROR:", e)
        log["errors"].append({"system": "linear", "error": str(e), "traceback": tb})

    # --- Freshdesk ---
    try:
        freshdesk_key = read_api_key(FRESHDESK_API_KEY_PATH)
        last_fd = read_last_snapshot(FRESHDESK_LAST_SNAPSHOT)
        if last_fd is None:
            cutoff_fd = utc_now_ts() - timedelta(days=BACKFILL_DAYS)
            first_run_fd = True
        else:
            cutoff_fd = last_fd
            first_run_fd = False

        print(f"[Freshdesk] cutoff = {cutoff_fd.isoformat()} (first_run={first_run_fd})")

        freshdesk_tickets = fetch_freshdesk_tickets(freshdesk_key, cutoff_dt=cutoff_fd)
        print(f"[Freshdesk] fetched {len(freshdesk_tickets)} tickets (post-filter)")

        raw_fd_path = FRESHDESK_RAW_SUBDIR / f"freshdesk_raw_{ts}.json"
        atomic_write_text(raw_fd_path, json.dumps(freshdesk_tickets, indent=2, ensure_ascii=False))
        print(f"[Freshdesk] saved raw JSON to {raw_fd_path}")

        fd_rows = [normalize_freshdesk_ticket(t, snapshot_ts_iso, raw_fd_path) for t in freshdesk_tickets]
        df_fd = pd.DataFrame(fd_rows)
        if not df_fd.empty:
            append_csv_atomic(DATA_FRESHDESK_CSV, df_fd)
            print(f"[Freshdesk] appended {len(df_fd)} rows to {DATA_FRESHDESK_CSV}")
        else:
            print("[Freshdesk] no rows to append")

        write_last_snapshot(FRESHDESK_LAST_SNAPSHOT, start_time)
        log["freshdesk"]["count"] = len(freshdesk_tickets)
    except Exception as e:
        tb = traceback.format_exc()
        print("[Freshdesk] ERROR:", e)
        log["errors"].append({"system": "freshdesk", "error": str(e), "traceback": tb})

    # Save run log
    log_path = LOGS_DIR / f"snapshot_log_{ts}.json"
    atomic_write_text(log_path, json.dumps(log, indent=2, ensure_ascii=False))
    print("Snapshot run complete. Log saved to", log_path)
    return log

# -------------------------
# Run
# -------------------------
if __name__ == "__main__":
    try:
        result = run_snapshot()
        print("Result:", json.dumps(result, indent=2, ensure_ascii=False))
    except Exception as e:
        print("Fatal error during snapshot:", e)
        print(traceback.format_exc())
        sys.exit(1)
